In [1]:
import pandas as pd
BASE_DIR = r"C:\Users\s2\Documents\Federated\dataset"

DATA_PATH = (
    f"{BASE_DIR}/TUANDROMD 2.csv"
)
data = pd.read_csv(DATA_PATH)
data

,ACCESS_ALL_DOWNLOADS,ACCESS_CACHE_FILESYSTEM,ACCESS_CHECKIN_PROPERTIES,ACCESS_COARSE_LOCATION,ACCESS_COARSE_UPDATES,ACCESS_FINE_LOCATION,ACCESS_LOCATION_EXTRA_COMMANDS,ACCESS_MOCK_LOCATION,ACCESS_MTK_MMHW,ACCESS_NETWORK_STATE,...,Landroid/telephony/TelephonyManager;->getLine1Number,Landroid/telephony/TelephonyManager;->getNetworkOperator,Landroid/telephony/TelephonyManager;->getNetworkOperatorName,Landroid/telephony/TelephonyManager;->getNetworkCountryIso,Landroid/telephony/TelephonyManager;->getSimOperator,Landroid/telephony/TelephonyManager;->getSimOperatorName,Landroid/telephony/TelephonyManager;->getSimCountryIso,Landroid/telephony/TelephonyManager;->getSimSerialNumber,Lorg/apache/http/impl/client/DefaultHttpClient;->execute,Label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,malware
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,malware
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,malware
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,malware
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,malware
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4460,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,goodware
4461,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,goodware
4462,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,goodware
4463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,goodware


In [2]:
# Cek jumlah missing value awal
print(data.isna().sum())

# Hapus baris dengan missing value dan simpan hasilnya
data = data.dropna()

# Cek kembali jumlah missing value setelah dihapus
print(data.isna().sum())

ACCESS_ALL_DOWNLOADS                                        1
ACCESS_CACHE_FILESYSTEM                                     1
ACCESS_CHECKIN_PROPERTIES                                   1
ACCESS_COARSE_LOCATION                                      1
ACCESS_COARSE_UPDATES                                       1
                                                           ..
Landroid/telephony/TelephonyManager;->getSimOperatorName    1
Landroid/telephony/TelephonyManager;->getSimCountryIso      1
Landroid/telephony/TelephonyManager;->getSimSerialNumber    1
Lorg/apache/http/impl/client/DefaultHttpClient;->execute    1
Label                                                       1
Length: 242, dtype: int64
ACCESS_ALL_DOWNLOADS                                        0
ACCESS_CACHE_FILESYSTEM                                     0
ACCESS_CHECKIN_PROPERTIES                                   0
ACCESS_COARSE_LOCATION                                      0
ACCESS_COARSE_UPDATES                       

In [3]:
data['Label'].value_counts()

Label
malware     3565
goodware     899
Name: count, dtype: int64

In [4]:
# Mapping label
data["Label"] = data["Label"].map({
    "goodware": 0,
    "malware": 1
})

# Cek distribusi kelas
print(data["Label"].value_counts())

Label
1    3565
0     899
Name: count, dtype: int64


In [5]:
# Menghitung jumlah kolom per tipe data
print(data.dtypes.value_counts())

float64    241
int64        1
Name: count, dtype: int64


In [6]:
# Memisahkan Fitur (X) dan Label (y)
X = data.drop(columns=["Label"])
y = data["Label"]

# Cek dimensi/ukuran data hasil pemisahan
print("Ukuran Fitur (X):", X.shape)
print("Ukuran Label (y):", y.shape)

Ukuran Fitur (X): (4464, 241)
Ukuran Label (y): (4464,)


In [7]:
from dataset import prepare_dataset

# 3. Panggil pipeline
data = prepare_dataset(X, y, val_size=0.15, test_size=0.15, task="classification")

X_train_t, y_train_t = data["train"]
X_val_t, y_val_t     = data["val"]
X_test_t, y_test_t   = data["test"]


Train : torch.Size([3124, 241]) torch.Size([3124])
Val   : torch.Size([670, 241]) torch.Size([670])
Test  : torch.Size([670, 241]) torch.Size([670])


In [8]:
from dataset import prepare_dataset
from model import build_model
from training import train_model
from evaluate import evaluate_model

data = prepare_dataset(X, y, task="classification")
in_dim = data["train"][0].shape[1]
model = build_model(in_dim=in_dim, num_classes=2)

train_model(model, data, epochs=10, verbose=False)

metrics = evaluate_model(model, data["test"],)

Train : torch.Size([3124, 241]) torch.Size([3124])
Val   : torch.Size([670, 241]) torch.Size([670])
Test  : torch.Size([670, 241]) torch.Size([670])
Accuracy  : 0.9881
Precision : 0.9789
Recall    : 0.9842
F1-score  : 0.9815

Confusion Matrix:
[[132   3]
 [  5 530]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       135
           1       0.99      0.99      0.99       535

    accuracy                           0.99       670
   macro avg       0.98      0.98      0.98       670
weighted avg       0.99      0.99      0.99       670

